# Benchmark: Parallel vs Sequential Explorer

Measures wall time for the same experiment workload under:
- Sequential execution (`parallel=False`)
- Parallel execution with varying worker counts

Also verifies correctness: both modes must produce identical results.

## Workload
CUBIC_PANCAKE, group `"different"`, n=4–12, subset 1–7 → 7 × 3 cosets × 9 n-values = **189 experiments**.
Adjust `BENCH_MAX_N` and `BENCH_SUBSET_RANGE` below to tune size.

## Cell 1: Imports and Setup

In [1]:
import sys
import os
import time
import shutil

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'Utils'))

from explorer import Explorer, CUBIC_PANCAKE
import pandas as pd
import plotly.express as px

print(f"CPUs available: {os.cpu_count()}")

ModuleNotFoundError: No module named 'cayleypy'

## Cell 2: Workload Configuration

Change these to tune benchmark size. Larger values make the speedup more visible
but take longer to run.

In [ ]:
BENCH_GROUP      = "different"     # coset group to benchmark
BENCH_MIN_N      = 4
BENCH_MAX_N      = 12              # keep ≤ 14 for a reasonable wall time
BENCH_SUBSET_RANGE = (1, 7)        # sweep all 7 cubic-pancake subsets

# Separate output dirs so sequential and parallel never share a cache
DIR_SEQ = "bench_output/sequential"
DIR_PAR = "bench_output/parallel"

n_combinations = (
    (BENCH_SUBSET_RANGE[1] - BENCH_SUBSET_RANGE[0] + 1)  # subsets
    * 3                                                    # cosets in 'different'
    * (BENCH_MAX_N - BENCH_MIN_N + 1)                     # n values
)
print(f"Total experiments: {n_combinations}")

## Cell 3: Sequential Benchmark

`parallel=False` — single process, experiments run one at a time.

In [ ]:
# Fresh dir each time so skip_computed never hides work
shutil.rmtree(DIR_SEQ, ignore_errors=True)

exp_seq = Explorer(
    config=CUBIC_PANCAKE,
    output_dir=DIR_SEQ,
    min_n=BENCH_MIN_N,
    max_n=BENCH_MAX_N,
)

t0 = time.perf_counter()
df_seq = exp_seq.run_and_save(
    BENCH_GROUP,
    parallel=False,
    max_n=BENCH_MAX_N,
    subset_range=BENCH_SUBSET_RANGE,
    plot=False,
)
t_seq = time.perf_counter() - t0

print(f"\nSequential wall time: {t_seq:.2f}s  ({len(df_seq)} rows)")

## Cell 4: Parallel Benchmark (all available CPUs)

`parallel=True` with `max_workers=os.cpu_count()`.

In [ ]:
shutil.rmtree(DIR_PAR, ignore_errors=True)

exp_par = Explorer(
    config=CUBIC_PANCAKE,
    output_dir=DIR_PAR,
    min_n=BENCH_MIN_N,
    max_n=BENCH_MAX_N,
)

n_workers_max = os.cpu_count()

t0 = time.perf_counter()
df_par = exp_par.run_and_save(
    BENCH_GROUP,
    parallel=True,
    max_workers=n_workers_max,
    max_n=BENCH_MAX_N,
    subset_range=BENCH_SUBSET_RANGE,
    plot=False,
)
t_par = time.perf_counter() - t0

print(f"\nParallel ({n_workers_max} workers) wall time: {t_par:.2f}s  ({len(df_par)} rows)")
print(f"Speedup vs sequential: {t_seq / t_par:.2f}x")

## Cell 5: Correctness Check

Sort both DataFrames by the same key columns and compare `diameter` values.
Any mismatch indicates a bug in the parallel implementation.

In [ ]:
key_cols = ["coset", "subset", "n"]

def canonical(df):
    return (
        df[key_cols + ["diameter", "total_states"]]
        .sort_values(key_cols)
        .reset_index(drop=True)
    )

canon_seq = canonical(df_seq)
canon_par = canonical(df_par)

if canon_seq.equals(canon_par):
    print("PASS: Sequential and parallel results are identical.")
else:
    diff = canon_seq.compare(canon_par)
    print(f"FAIL: {len(diff)} differing rows:")
    display(diff)

print(f"\nRow counts — sequential: {len(df_seq)}, parallel: {len(df_par)}")

## Cell 6: Worker-Count Scaling

Sweep `max_workers` from 1 up to `os.cpu_count()` to see how wall time scales.
Each run uses `skip_computed=False` and a fresh directory.

In [ ]:
import math

cpu_count = os.cpu_count() or 1

# Build a deduplicated list: 1, 2, 4, 8, ..., cpu_count
worker_counts = sorted(set(
    [1, 2]
    + [2**i for i in range(2, int(math.log2(cpu_count)) + 1) if 2**i <= cpu_count]
    + [cpu_count]
))
print(f"Worker counts to test: {worker_counts}")

scaling_results = []

for n_workers in worker_counts:
    run_dir = f"bench_output/scale_{n_workers}"
    shutil.rmtree(run_dir, ignore_errors=True)

    exp = Explorer(
        config=CUBIC_PANCAKE,
        output_dir=run_dir,
        min_n=BENCH_MIN_N,
        max_n=BENCH_MAX_N,
    )

    if n_workers == 1:
        t0 = time.perf_counter()
        exp.run_and_save(
            BENCH_GROUP,
            parallel=False,
            max_n=BENCH_MAX_N,
            subset_range=BENCH_SUBSET_RANGE,
            plot=False,
        )
    else:
        t0 = time.perf_counter()
        exp.run_and_save(
            BENCH_GROUP,
            parallel=True,
            max_workers=n_workers,
            max_n=BENCH_MAX_N,
            subset_range=BENCH_SUBSET_RANGE,
            plot=False,
        )

    elapsed = time.perf_counter() - t0
    scaling_results.append({"workers": n_workers, "wall_time_s": elapsed})
    print(f"  workers={n_workers:3d}  time={elapsed:.2f}s")

df_scaling = pd.DataFrame(scaling_results)
t_baseline = df_scaling.loc[df_scaling["workers"] == 1, "wall_time_s"].values[0]
df_scaling["speedup"] = t_baseline / df_scaling["wall_time_s"]
df_scaling["efficiency"] = df_scaling["speedup"] / df_scaling["workers"]
display(df_scaling)

## Cell 7: Speedup Chart

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

# Measured wall time
fig.add_trace(go.Bar(
    x=df_scaling["workers"].astype(str),
    y=df_scaling["wall_time_s"],
    name="Wall time (s)",
    yaxis="y1",
))

# Measured speedup
fig.add_trace(go.Scatter(
    x=df_scaling["workers"].astype(str),
    y=df_scaling["speedup"],
    name="Speedup (actual)",
    mode="lines+markers",
    yaxis="y2",
))

# Ideal linear speedup reference
fig.add_trace(go.Scatter(
    x=df_scaling["workers"].astype(str),
    y=df_scaling["workers"].astype(float),
    name="Ideal linear speedup",
    mode="lines",
    line=dict(dash="dash"),
    yaxis="y2",
))

fig.update_layout(
    title=f"Parallel scaling — {BENCH_GROUP} group, n={BENCH_MIN_N}–{BENCH_MAX_N}, "
          f"subsets {BENCH_SUBSET_RANGE[0]}–{BENCH_SUBSET_RANGE[1]}",
    xaxis_title="Number of workers",
    yaxis=dict(title="Wall time (s)", side="left"),
    yaxis2=dict(title="Speedup (×)", side="right", overlaying="y"),
    legend=dict(x=0.5, y=1.12, orientation="h"),
)
fig.show()

## Cell 8: Cleanup (optional)

Remove all benchmark output directories. Run this after you're done inspecting results.

In [ ]:
# Uncomment to delete all benchmark artifacts
# shutil.rmtree("bench_output", ignore_errors=True)
# print("Cleaned up bench_output/")